In [38]:
from __future__ import annotations

# ===== Standard Library =====
import os
import re
import math
import argparse
from typing import List, Tuple, Dict, Iterable
from collections import Counter, defaultdict
from itertools import combinations

# ===== Third-Party =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as pltsS
from tqdm import tqdm
from scipy.optimize import linear_sum_assignment

# ===== scikit-learn =====
# Data
from sklearn.datasets import fetch_20newsgroups

# Text / Features
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Dimensionality Reduction
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.manifold import TSNE

# Clustering
from sklearn.cluster import KMeans

# Metrics
from sklearn.metrics import (
    f1_score,
    adjusted_rand_score,
    homogeneity_score,
    confusion_matrix,
    accuracy_score,
    recall_score,
    classification_report,
)

# Preprocessing
from sklearn.preprocessing import normalize

from dotenv import load_dotenv
load_dotenv()

True

In [39]:
df = pd.read_csv('/home/ys0660/2507Sub/textclustering/0915/stage1/data/20ng.csv')

In [3]:
df.head()

,Unnamed: 0,text,label_id,label_name,cleaned_text
0,0,i am sure some bashers of pens fans are pretty...,10,rec.sport.hockey,sure bashers pens fans pretty confused lack ki...
1,1,my brother is in the market for a high perform...,3,comp.sys.ibm.pc.hardware,brother market high performance video card sup...
2,2,the student of regional killings alias davidia...,17,talk.politics.mideast,student regional killings alias davidian david...
3,3,in article wayne smith writes think it s the s...,3,comp.sys.ibm.pc.hardware,wayne smith writes think scsi card doing dma t...
4,4,1 i have an old jasmine drive which i cannot u...,4,comp.sys.mac.hardware,old jasmine drive use new understanding upsate...


In [11]:
cleaned_text = df['cleaned_text'].tolist()
y_true = df['label_id'].tolist()

In [17]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [18]:
all_vecs = []
batch_size = 128  # 배치 크기 조정 가능
model_name = "text-embedding-3-small"

In [19]:
import tiktoken

enc = tiktoken.encoding_for_model("text-embedding-3-small")

MAX_TOKENS = 8192
def truncate_text(text, max_tokens=MAX_TOKENS):
    tokens = enc.encode(text)
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
    return enc.decode(tokens)

In [31]:
print("[2] Call API for Embeddings...")

def get_embed_dim(model_name: str) -> int:
    name = model_name.lower()
    if "text-embedding-3-large" in name:
        return 3072
    # small/ada-002 등 기본 1536
    return 1536

def embed_texts_and_align_labels(
    texts: List[Union[str, List[str]]],   # string 또는 tokenized list
    labels: List[int],
    model: str = "text-embedding-3-small"
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[str, List[int]]]:
    """
    반환:
      X: (N, D)  - 입력 개수 그대로 유지(빈/실패는 0-벡터)
      mask_nonzero: (N,) - 0-벡터가 아닌 행(True)
      labels_clean: (M,) - mask_nonzero 적용
      texts_clean:  (M,) - mask_nonzero 적용
      logs: dict    - {'ok': [...], 'empty': [...], 'fail': [...], 'dim': D}
    """
    assert len(texts) == len(labels), f"len(texts)={len(texts)} vs len(labels)={len(labels)}"
    N = len(texts)
    D = get_embed_dim(model)

    X = np.zeros((N, D), dtype=np.float32)
    mask_nonzero = np.zeros(N, dtype=bool)

    ok_idx: List[int] = []
    empty_idx: List[int] = []
    fail_idx: List[int] = []

    for idx, raw in enumerate(tqdm(texts, desc="Embedding", total=N)):
        # --- [변경된 부분] 리스트면 문자열로 합치기 ---
        if isinstance(raw, list):
            text = " ".join(map(str, raw))   # 리스트 → 문자열
        else:
            text = str(raw)

        text = truncate_text(text.strip())  # 토큰 제한 및 공백 제거

        if not text:
            empty_idx.append(idx)
            continue
        try:
            resp = client.embeddings.create(model=model, input=text)
            vec = np.asarray(resp.data[0].embedding, dtype=np.float32)

            # 차원 안전장치
            if vec.shape[0] != D:
                print(f"[warn] idx={idx} dim change: {vec.shape[0]} != {D}. Adjusting matrix.")
                newD = vec.shape[0]
                if newD > D:
                    pad = np.zeros((N, newD - D), dtype=np.float32)
                    X = np.hstack([X, pad])
                else:
                    X = X[:, :newD]
                D = newD

            X[idx] = vec
            mask_nonzero[idx] = True
            ok_idx.append(idx)
        except Exception as e:
            print(f"[warn] idx={idx} embedding failed: {e!r}")
            fail_idx.append(idx)

    # 요약 로그
    print("\n=== Embedding Summary ===")
    print(f"Total inputs   : {N}")
    print(f"OK             : {len(ok_idx)}")
    print(f"Empty texts    : {len(empty_idx)}")
    print(f"API failures   : {len(fail_idx)}")
    print(f"Kept (non-zero): {mask_nonzero.sum()} | Dropped: {(~mask_nonzero).sum()}")
    if empty_idx[:5]:
        print(f"First empty idx: {empty_idx[:5]}")
    if fail_idx[:5]:
        print(f"First fail idx : {fail_idx[:5]}")

    labels_arr = np.asarray(labels)
    texts_arr  = np.asarray(texts, dtype=object)
    labels_clean = labels_arr[mask_nonzero]
    texts_clean  = texts_arr[mask_nonzero]

    logs = {"ok": ok_idx, "empty": empty_idx, "fail": fail_idx, "dim": D}
    return X, mask_nonzero, labels_clean, texts_clean, logs


[2] Call API for Embeddings...


In [33]:
def save_clean_pack(save_path: str, X_clean: np.ndarray,
                    labels_clean: np.ndarray, texts_clean: np.ndarray):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    np.savez(save_path, X=X_clean, labels=labels_clean, texts=texts_clean)
    print(f"[saved] {save_path} | X={X_clean.shape}, labels={labels_clean.shape}, texts={texts_clean.shape}")

In [36]:
X, mask_nonzero, labels_clean, texts_clean, logs = embed_texts_and_align_labels(
    cleaned_text, y_true, model="text-embedding-3-small"
)
X_clean = X[mask_nonzero]  # 여기서만 마스크 적용

print("Before/After:", X.shape, "->", X_clean.shape)

# 저장
save_path = "/home/ys0660/2507Sub/textclustering/0915/stage1/01data/mine_gpt_embedding.npz"
save_clean_pack(save_path, X_clean, labels_clean, texts_clean)

Embedding: 100%|██████████| 18700/18700 [2:03:23<00:00,  2.53it/s]  


=== Embedding Summary ===
Total inputs   : 18700
OK             : 18700
Empty texts    : 0
API failures   : 0
Kept (non-zero): 18700 | Dropped: 0
Before/After: (18700, 1536) -> (18700, 1536)
[saved] /home/ys0660/2507Sub/textclustering/0915/stage1/01data/mine_gpt_embedding.npz | X=(18700, 1536), labels=(18700,), texts=(18700,)
